# 02_raw_to_bronze
Reads one landing file with an explicit schema, adds metadata and row_hash, keeps only new or changed rows on incremental loads, and MERGEs into bronze_steam.
Parameters: load_type (full / incremental), date (YYYY-MM-DD), landing_path (optional override).

In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("date", "")
dbutils.widgets.text("landing_path", "")

In [0]:
import uuid
from datetime import datetime, date as _date
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

CATALOG, SCHEMA = "workspace", "steam"
LANDING_DIR = "/Volumes/workspace/default/steam"
LAYER = "Raw-to-Bronze"
spark.sql(f"USE {CATALOG}.{SCHEMA}")

load_type = dbutils.widgets.get("load_type")
run_date = dbutils.widgets.get("date").strip() or str(_date.today())
datetime.strptime(run_date, "%Y-%m-%d")  # fails fast on a bad date
landing_path = dbutils.widgets.get("landing_path").strip() or f"{LANDING_DIR}/steamspy_{run_date}.jsonl"
batch_id = str(uuid.uuid4())
print(load_type, run_date, landing_path, batch_id)

In [0]:
SOURCE_COLS = ["appid", "name", "developer", "publisher", "score_rank", "userscore",
               "positive", "negative", "owners", "average_forever", "average_2weeks",
               "median_forever", "median_2weeks", "price", "initialprice", "discount", "ccu"]

def build_schema(extra_cols=()):
    return StructType([StructField(c, StringType(), True) for c in SOURCE_COLS + list(extra_cols)])

In [0]:
def detect_extra_columns(path):
    """Keys present in the file but not in the expected schema."""
    keys = (spark.read.text(path)
            .select(F.explode(F.map_keys(F.from_json("value", "map<string,string>"))).alias("k"))
            .distinct().collect())
    return sorted({r.k for r in keys} - set(SOURCE_COLS))

def log_new_columns(cols, path):
    """Record columns that Bronze did not have before this run."""
    if not cols:
        return
    (spark.createDataFrame([(c,) for c in cols], "column_name string")
        .select("column_name",
                F.lit("new_column").alias("drift_type"),
                F.to_date(F.lit(run_date)).alias("detected_on"),
                F.lit(path).alias("source_file"),
                F.current_timestamp().alias("load_timestamp"))
        .write.mode("append").saveAsTable("schema_drift_log"))

In [0]:
def read_landing(path, extra_cols):
    df = spark.read.schema(build_schema(extra_cols)).json(path)
    hash_input = F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("")) for c in SOURCE_COLS])
    return (df.dropDuplicates(["appid"])
              .withColumn("source_file", F.lit(path))
              .withColumn("batch_id", F.lit(batch_id))
              .withColumn("load_type", F.lit(load_type))
              .withColumn("load_date", F.to_date(F.lit(run_date)))
              .withColumn("row_hash", F.sha2(hash_input, 256))
              .withColumn("load_timestamp", F.current_timestamp()))

In [0]:
def filter_changed(df):
    """Compare each row's hash with that appid's latest Bronze hash from earlier dates."""
    w = Window.partitionBy("appid").orderBy(F.col("load_date").desc())
    baseline = (spark.table("bronze_steam")
                .where(F.col("load_date") < F.to_date(F.lit(run_date)))
                .withColumn("rn", F.row_number().over(w))
                .where("rn = 1")
                .select(F.col("appid").alias("b_appid"), F.col("row_hash").alias("b_hash")))
    return (df.join(baseline, df.appid == baseline.b_appid, "left")
              .where(F.col("b_hash").isNull() | (F.col("b_hash") != F.col("row_hash")))
              .drop("b_appid", "b_hash"))

In [0]:
def merge_into_bronze(df):
    df.createOrReplaceTempView("bronze_updates")
    res = spark.sql("""
        MERGE WITH SCHEMA EVOLUTION INTO bronze_steam t
        USING bronze_updates s
        ON t.appid <=> s.appid AND t.load_date = s.load_date
        WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    return res["num_inserted_rows"], res["num_updated_rows"]

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("load_type", StringType()), StructField("file_processed", StringType()),
    StructField("batch_id", StringType()), StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()), StructField("status", StringType()),
    StructField("rows_inserted", LongType()), StructField("rows_updated", LongType()),
    StructField("rows_skipped_unchanged", LongType()), StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType())])

def write_log(start, status, inserted=None, updated=None, skipped=None, error=None):
    row = (str(uuid.uuid4()), LAYER, load_type, landing_path, batch_id, start, datetime.now(),
           status, inserted, updated, skipped, 0 if status == "Success" else None, error)
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").saveAsTable("pipeline_execution_logs")

In [0]:
start = datetime.now()
try:
    extra_cols = detect_extra_columns(landing_path)
    new_for_bronze = [c for c in extra_cols if c not in spark.table("bronze_steam").columns]

    df = read_landing(landing_path, extra_cols)
    total = df.count()
    to_load = filter_changed(df) if load_type == "incremental" else df
    kept = to_load.count()

    inserted, updated = merge_into_bronze(to_load)
    log_new_columns(new_for_bronze, landing_path)
    write_log(start, "Success", inserted, updated, total - kept)
    print(f"file rows: {total} | inserted: {inserted} | updated: {updated} | skipped: {total - kept}")
except Exception as e:
    write_log(start, "Failure", error=str(e)[:2000])
    raise

In [0]:
display(spark.sql("SELECT load_date, load_type, COUNT(*) AS n FROM bronze_steam GROUP BY 1, 2 ORDER BY 1"))
display(spark.sql("SELECT * FROM pipeline_execution_logs ORDER BY start_time DESC LIMIT 5"))